# Pre-Processing


## Crawling

### Install Liberary

In [1]:
!pip install -q nltk contractions emoji pyspellchecker Sastrawi


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


### Liberary

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, sys, time
import string
import nltk
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import contractions
import emoji
from spellchecker import SpellChecker
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm

### Download tokenizer


In [34]:
nltk.download("punkt")
nltk.download("punkt_tab")

# Download stopwords
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Fungsi Crawling

In [1]:
# === BASE URL PTA Trunojoyo ===
BASE_URL = "https://pta.trunojoyo.ac.id/c_search/byprod"

def get_max_page(prodi_id):
    url = f"{BASE_URL}/{prodi_id}/1"
    r = requests.get(url)
    soup = BeautifulSoup(r.content, "html.parser")

    # Cari tombol last page
    last_page = soup.select_one('ol.pagination a:contains("»")')
    if last_page and "href" in last_page.attrs:
        href = last_page["href"]
        max_page = int(href.split("/")[-1])
        return max_page

    # fallback kalau pagination tidak ada
    return 1

# === Progress bar sederhana ===
def print_progress(prodi, current, total):
    progress = (current / total) * 100
    sys.stdout.write(
        f"\r📖 Prodi {prodi} - Halaman {current}/{total} [{progress:.1f}%]"
    )
    sys.stdout.flush()

# === Fungsi utama untuk scraping PTA Trunojoyo ===
def pta_manajemen():
    start_time = time.time()

    data = {
        "id": [],
        "penulis": [],
        "judul": [],
        "abstrak_id": [],
        "abstrak_en": [],
        "pembimbing_pertama": [],
        "pembimbing_kedua": [],
        "prodi": []
    }

    # === Ganti sesuai ID prodi Manajemen (contoh: 35) ===
    prodi_id = 7

    # cari jumlah halaman prodi manajemen
    max_page = get_max_page(prodi_id)

    for j in range(1, max_page + 1):
        url = f"{BASE_URL}/{prodi_id}/{j}"
        r = requests.get(url)
        soup = BeautifulSoup(r.content, "html.parser")
        jurnals = soup.select('li[data-cat="#luxury"]')

        isii = soup.select_one('div#begin')
        if not isii:
            continue
        prodi_full = isii.select_one('h2').text.strip()
        prodi = prodi_full.replace("Journal Jurusan ", "")

        for jurnal in jurnals:
            link_keluar = jurnal.select_one('a.gray.button')['href']

            # ambil ID dari link PTA
            id_match = re.search(r"/detail/(\d+)", link_keluar)
            pta_id = id_match.group(1) if id_match else None

            response = requests.get(link_keluar)
            soup1 = BeautifulSoup(response.content, "html.parser")
            isi = soup1.select_one('div#content_journal')

            if not isi:
                continue

            judul = isi.select_one('a.title').text.strip()
            penulis = isi.select_one('span:contains("Penulis")').text.split(' : ')[1]
            pembimbing_pertama = isi.select_one('span:contains("Dosen Pembimbing I")').text.split(' : ')[1]
            pembimbing_kedua = isi.select_one('span:contains("Dosen Pembimbing II")').text.split(' :')[1]

            paragraf = isi.select('p[align="justify"]')
            abstrak_id = paragraf[0].get_text(strip=True) if len(paragraf) > 0 else "N/A"
            abstrak_en = paragraf[1].get_text(strip=True) if len(paragraf) > 1 else "N/A"

            data["id"].append(pta_id)
            data["penulis"].append(penulis)
            data["judul"].append(judul)
            data["abstrak_id"].append(abstrak_id)
            data["abstrak_en"].append(abstrak_en)
            data["pembimbing_pertama"].append(pembimbing_pertama)
            data["pembimbing_kedua"].append(pembimbing_kedua)
            data["prodi"].append(prodi)

        # update progress
        print_progress(prodi, j, max_page)

    sys.stdout.write("\n")

    # simpan ke CSV
    df = pd.DataFrame(data)
    df.to_csv("pta_manajemen.csv", index=False, encoding="utf-8-sig")

    # hitung durasi
    end_time = time.time()
    elapsed = int(end_time - start_time)
    jam, sisa = divmod(elapsed, 3600)
    menit, detik = divmod(sisa, 60)

    # summary
    print("\n✅ Seluruh data Manajemen berhasil dikumpulkan!")
    print(f"📊 Total entri: {len(df)}")
    print(f"⏱️ Waktu eksekusi: {jam} jam {menit} menit {detik} detik")

    return df

# === Jalankan scraper ===
if __name__ == "__main__":
    pta_manajemen()


/usr/local/lib/python3.12/dist-packages/soupsieve/css_parser.py:876: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028


📖 Prodi Manajemen - Halaman 207/207 [100.0%]

✅ Seluruh data Manajemen berhasil dikumpulkan!
📊 Total entri: 1031
⏱️ Waktu eksekusi: 0 jam 30 menit 14 detik


In [19]:
pta_manajemen_df= pd.read_csv("pta_manajemen.csv")
pta_manajemen_df

,id,penulis,judul,abstrak_id,abstrak_en,pembimbing_pertama,pembimbing_kedua,prodi
0,80211100070,SATIYAH,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",ABSTRACT\r\n\r\nIn an effort to increase labor...,"Dra. Hj. S. Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST.SE,M.MT",Manajemen
1,90211200001,Faishal,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Tujuan penelitian ini adalah untuk mengetahui ...,This study wanted to know the brand associatio...,Nurita Andriani,Yustina Chrismardani,Manajemen
2,80211100050,Wahyu Kurniawan,PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...,NaN,NaN,"Dr. Dra. Hj. Iriani Ismail, MM","Dra. Hj. S. Anugrahini Irawati, MM",Manajemen
3,100211200002,Muhammad Zakaria Utomo,Pengukuran Website Quality Pada Situs Sistem A...,Aplikasi nyata pemanfaatan teknologi informasi...,Academic portal system in University of Trunoj...,"Dr. Ir. Nurita Andriani, MM","Nirma Kurriwati, SP, M.Si",Manajemen
4,80211100044,Hendri Wahyudi Prayitno,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,Abstrak\r\nPenelitian ini menggunakan metode k...,Abstract\r\nThis research use quantitative met...,"Dra. Hj. S Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST,SE,.MT",Manajemen
...,...,...,...,...,...,...,...,...
1026,160211100071,Husnul Hotimah,Analisis Cost Volume Profit Untuk Menentukan T...,ABSTRAK\nPenelitian ini bertujuan untuk menget...,ABSTRACT\nThis study aims to determine the cal...,"Hj. Evaliati Amaniyah, S.E., M.S.M.",NaN,Manajemen
1027,160211100291,Uswatun Hasanah,Pengaruh Pelatihan Dan Kompensasi Terhadap Pro...,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...","ABSTRACT\nUswatun Hasanah, 160211100291, The E...","Dr. Raden Mas Mochammad Wispandono S.E ., MS",NaN,Manajemen
1028,160211100064,ACH FATHONI,PERAN SERVICE PERFORMANCE DAN CLIMATE ORGANIZA...,ABSTRAK\nTujuan dari penelitian ini adalah unt...,ABSTRACK\n The purpose of this study...,"YUDHI PRASETYA MADA, S.E., M.M.",NaN,Manajemen
1029,160211100030,INTAN YULLIA NINGSIH,BAURAN PROMOSI PADA DEALER YAMAHA TRETAN MOTOR...,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,ABSTRACK\nThis study aims: (1) To find out whe...,"DR. MOHAMMAD ARIEF, S.E., M.M.",NaN,Manajemen


## Fungsi cleaning dengan emoji


In [26]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()  # huruf kecil
    text = emoji.demojize(text)  # ubah emoji jadi teks, misal 😀 -> :grinning_face:
    text = re.sub(r'\d+', '', text)  # hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation))  # hapus tanda baca
    text = re.sub(r'\W', ' ', text)  # hapus karakter non-kata
    text = BeautifulSoup(text, "html.parser").get_text()  # hapus tag HTML
    text = re.sub(r'\s+', ' ', text).strip()  # normalisasi spasi
    return text

# Baca CSV
pta_manajemen_df = pd.read_csv("pta_manajemen.csv", dtype=str).fillna("")

# Terapkan langsung ke kolom target
pta_manajemen_df["abstrak_id_clean"] = pta_manajemen_df["abstrak_id"].apply(clean_text)

# Cleaning kolom target
pta_manajemen_df["abstrak_id_clean"] = pta_manajemen_df["abstrak_id"].apply(clean_text)

In [29]:
print("\nPTA (abstrak_id):")
pta_manajemen_df[["abstrak_id", "abstrak_id_clean"]].head(10)


PTA (abstrak_id):


,abstrak_id,abstrak_id_clean
0,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",abstrak satiyah pengaruh faktorfaktor pelatiha...
1,Tujuan penelitian ini adalah untuk mengetahui ...,tujuan penelitian ini adalah untuk mengetahui ...
2,,
3,Aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata pemanfaatan teknologi informasi...
4,Abstrak\r\nPenelitian ini menggunakan metode k...,abstrak penelitian ini menggunakan metode kuan...
5,"Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi...",abstrak aththaariq pengaruh kompetensi dosen t...
6,"ABSTRAK\r\nHaryono Arifin, Pengaruh Perilaku K...",abstrak haryono arifin pengaruh perilaku konsu...
7,"ABSTRAK\r\n\tDharma Abidin Syah,Kesimpulan: (1...",abstrak dharma abidin syahkesimpulan terdapat ...
8,ABSTRAK\r\n\r\nTujuan penelitian ini adalah un...,abstrak tujuan penelitian ini adalah untuk men...
9,Hasil dari penelitian ini dari perhitungan Cre...,hasil dari penelitian ini dari perhitungan cre...


In [35]:
# Tokenisasi untuk PTA
pta_manajemen_df["abstrak_id_tokens"] = pta_manajemen_df["abstrak_id_clean"].apply(word_tokenize)

In [37]:
print("\nPTA (abstrak_id_tokens):")
pta_manajemen_df[["abstrak_id_clean", "abstrak_id_tokens"]].head(10)


PTA (abstrak_id_tokens):


,abstrak_id_clean,abstrak_id_tokens
0,abstrak satiyah pengaruh faktorfaktor pelatiha...,"[abstrak, satiyah, pengaruh, faktorfaktor, pel..."
1,tujuan penelitian ini adalah untuk mengetahui ...,"[tujuan, penelitian, ini, adalah, untuk, menge..."
2,,[]
3,aplikasi nyata pemanfaatan teknologi informasi...,"[aplikasi, nyata, pemanfaatan, teknologi, info..."
4,abstrak penelitian ini menggunakan metode kuan...,"[abstrak, penelitian, ini, menggunakan, metode..."
5,abstrak aththaariq pengaruh kompetensi dosen t...,"[abstrak, aththaariq, pengaruh, kompetensi, do..."
6,abstrak haryono arifin pengaruh perilaku konsu...,"[abstrak, haryono, arifin, pengaruh, perilaku,..."
7,abstrak dharma abidin syahkesimpulan terdapat ...,"[abstrak, dharma, abidin, syahkesimpulan, terd..."
8,abstrak tujuan penelitian ini adalah untuk men...,"[abstrak, tujuan, penelitian, ini, adalah, unt..."
9,hasil dari penelitian ini dari perhitungan cre...,"[hasil, dari, penelitian, ini, dari, perhitung..."


## Stopwords

In [38]:
# Stopwords untuk bahasa Indonesia
stop_words_id = set(stopwords.words('indonesian'))

# Filter stopwords di PTA
pta_manajemen_df["abstrak_id_filtered"] = pta_manajemen_df["abstrak_id_tokens"].apply(
    lambda tokens: [word for word in tokens if word not in stop_words_id]
)

In [40]:
print("\nPTA (abstrak_id_filtered):")
pta_manajemen_df[["abstrak_id_tokens", "abstrak_id_filtered"]].head(10)


PTA (abstrak_id_filtered):


,abstrak_id_tokens,abstrak_id_filtered
0,"[abstrak, satiyah, pengaruh, faktorfaktor, pel...","[abstrak, satiyah, pengaruh, faktorfaktor, pel..."
1,"[tujuan, penelitian, ini, adalah, untuk, menge...","[tujuan, penelitian, persepsi, brand, associat..."
2,[],[]
3,"[aplikasi, nyata, pemanfaatan, teknologi, info...","[aplikasi, nyata, pemanfaatan, teknologi, info..."
4,"[abstrak, penelitian, ini, menggunakan, metode...","[abstrak, penelitian, metode, kuantitatif, men..."
5,"[abstrak, aththaariq, pengaruh, kompetensi, do...","[abstrak, aththaariq, pengaruh, kompetensi, do..."
6,"[abstrak, haryono, arifin, pengaruh, perilaku,...","[abstrak, haryono, arifin, pengaruh, perilaku,..."
7,"[abstrak, dharma, abidin, syahkesimpulan, terd...","[abstrak, dharma, abidin, syahkesimpulan, peng..."
8,"[abstrak, tujuan, penelitian, ini, adalah, unt...","[abstrak, tujuan, penelitian, mengidentifikasi..."
9,"[hasil, dari, penelitian, ini, dari, perhitung...","[hasil, penelitian, perhitungan, credit, risk,..."


In [43]:
factory = StemmerFactory()
indo_stemmer = factory.create_stemmer()

pta_manajemen_df["abstrak_id_stemmed"] = pta_manajemen_df["abstrak_id_filtered"].apply(
    lambda tokens: [indo_stemmer.stem(word) for word in tokens]
)

In [44]:
print("\nPTA - Stemming & Lemmatization (abstrak_id):")
pta_manajemen_df[["abstrak_id_filtered", "abstrak_id_stemmed"]].head(10)


PTA - Stemming & Lemmatization (abstrak_id):


,abstrak_id_filtered,abstrak_id_stemmed
0,"[abstrak, satiyah, pengaruh, faktorfaktor, pel...","[abstrak, satiyah, pengaruh, faktorfaktor, lat..."
1,"[tujuan, penelitian, persepsi, brand, associat...","[tuju, teliti, persepsi, brand, association, l..."
2,[],[]
3,"[aplikasi, nyata, pemanfaatan, teknologi, info...","[aplikasi, nyata, manfaat, teknologi, informas..."
4,"[abstrak, penelitian, metode, kuantitatif, men...","[abstrak, teliti, metode, kuantitatif, tekan, ..."
5,"[abstrak, aththaariq, pengaruh, kompetensi, do...","[abstrak, aththaariq, pengaruh, kompetensi, do..."
6,"[abstrak, haryono, arifin, pengaruh, perilaku,...","[abstrak, haryono, arifin, pengaruh, perilaku,..."
7,"[abstrak, dharma, abidin, syahkesimpulan, peng...","[abstrak, dharma, abidin, syahkesimpulan, peng..."
8,"[abstrak, tujuan, penelitian, mengidentifikasi...","[abstrak, tuju, teliti, identifikasi, variabel..."
9,"[hasil, penelitian, perhitungan, credit, risk,...","[hasil, teliti, hitung, credit, risk, ratio, t..."


## Fungsi expand kontraksi bahasa Indonesia


In [45]:
def expand_indonesian_contractions(text):
  contractions_dict = {
      "gak": "tidak", "ga": "tidak", "nggak": "tidak", "enggak": "tidak", "ngga": "tidak", "gk": "tidak", "tdk": "tidak", "tk": "tidak",
      "gue": "saya", "gw": "saya", "gua": "saya", "sy": "saya", "aq": "saya", "q": "saya", "ane": "saya",
      "lu": "kamu", "loe": "kamu", "lo": "kamu", "km": "kamu", "kmu": "kamu", "elu": "kamu",
      "dah": "sudah", "udah": "sudah", "sdh": "sudah", "udh": "sudah",
      "blm": "belum", "td": "tadi", "ntar": "nanti", "skr": "sekarang", "skrg": "sekarang", "skg": "sekarang",
      "kmrn": "kemarin", "kemrn": "kemarin", "kmarin": "kemarin",
      "aja": "saja", "aj": "saja", "sj": "saja",
      "nih": "ini", "nie": "ini", "ni": "ini", "tuh": "itu", "gtu": "begitu", "gitu": "begitu",
      "trs": "terus", "trus": "terus",
      "yg": "yang", "utk": "untuk", "dlm": "dalam", "dr": "dari", "dg": "dengan", "jd": "jadi", "jg": "juga",
      "krn": "karena", "tp": "tetapi", "tpi": "tetapi", "sm": "sama", "thd": "terhadap",
      "dll": "dan lain-lain", "dsb": "dan sebagainya", "dst": "dan seterusnya",
      "banget": "sekali", "bgt": "sekali", "sgt": "sangat", "sngt": "sangat",
      "lg": "sedang", "sdg": "sedang",
      "dl": "dulu", "pls": "tolong", "tolongin": "tolong", "plis": "tolong",
      "wkwk": "tertawa", "wkwkwk": "tertawa", "hehe": "tertawa kecil", "hihi": "tertawa kecil",
      "btw": "ngomong-ngomong", "imo": "menurut saya", "imho": "menurut saya", "cmiiw": "koreksi jika saya salah",
      "idk": "saya tidak tahu", "jk": "hanya bercanda",
      "ok": "baik", "oke": "baik", "okey": "baik", "sip": "baik",
      "ciyus": "serius", "serem": "menyeramkan",
      "kl": "kalau", "klo": "kalau", "klu": "kalau",
      "spy": "supaya", "spya": "supaya",
      "bbrp": "beberapa", "tsb": "tersebut", "trsbt": "tersebut",
      "dpt": "dapat", "bs": "bisa", "bsa": "bisa",
      "stlh": "setelah", "sblm": "sebelum",

      "mnj": "manajemen", "man": "manajemen", "mgt": "management",
      "org": "organisasi", "org2": "organisasi", "orgzt": "organisasi",
      "str": "struktur", "stkt": "struktur",
      "ldr": "leader", "ldrshp": "leadership", "pimp": "pimpinan", "pemimp": "pemimpin",
      "pln": "perencanaan", "renc": "perencanaan", "plan": "perencanaan",
      "orgz": "organizing", "orgzn": "organisasi",
      "dir": "directing", "pgn": "pengarahan",
      "cnt": "control", "cont": "control", "ctrl": "kontrol", "pengend": "pengendalian",
      "eff": "efisiensi", "effct": "efektivitas",
      "sdm": "sumber daya manusia", "hr": "human resource", "hrd": "human resource development",
      "res": "resource", "rsc": "resource", "sda": "sumber daya alam",
      "inv": "investasi", "invt": "investasi",
      "pmas": "pemasaran", "mkt": "marketing", "mktg": "marketing",
      "prod": "produksi", "prdks": "produksi",
      "fin": "finance", "keu": "keuangan", "akut": "akuntansi", "acct": "akuntansi",
      "ris": "risiko", "rsko": "risiko",
      "anal": "analisis", "eval": "evaluasi",
      "strtg": "strategi", "stg": "strategi",
      "ops": "operasi", "opr": "operasional", "oprs": "operasional",
      "bsc": "balanced scorecard", "swot": "analisis swot", "pest": "analisis pest",
      "csr": "corporate social responsibility", "gcn": "good corporate governance",
      "qm": "quality management", "iso": "standar iso",
      "kpi": "key performance indicator", "indik": "indikator",
      "knowl": "knowledge management", "kmgt": "knowledge management",
      "chg": "change management", "innv": "inovasi",
      "cnfl": "konflik", "cnflt": "konflik",
      "krj": "kerja", "tm": "tim", "tmwrk": "kerja sama tim",
      "cst": "cost", "faktorfaktor": "faktor", "bya": "biaya",
      "val": "nilai", "valu": "value",
      "proj": "proyek", "prjk": "proyek",

      "med": "medis", "obat2": "obat-obatan", "rs": "rumah sakit",
      "dok": "dokter", "drg": "dokter gigi", "prof": "profesor",
      "pt": "perguruan tinggi", "univ": "universitas", "fak": "fakultas",
      "skripsi": "skripsi", "tesis": "tesis", "disertasi": "disertasi",
      "mhs": "mahasiswa", "mhsw": "mahasiswa"
  }


  pattern = r'\b(' + '|'.join(re.escape(key) for key in contractions_dict.keys()) + r')\b'

  def replace_match(match):
      return contractions_dict[match.group(0).lower()]

  expanded_text = re.sub(pattern, replace_match, text, flags=re.IGNORECASE)
  return expanded_text


pta_manajemen_df["abstrak_id_expanded"] = pta_manajemen_df["abstrak_id_stemmed"].apply(
    lambda tokens: expand_indonesian_contractions(" ".join(tokens)).split()
)


In [49]:
print("\nPTA - Abstrak ID (expanded):")
pta_manajemen_df[["abstrak_id_stemmed", "abstrak_id_expanded"]].head(10)


PTA - Abstrak ID (expanded):


,abstrak_id_stemmed,abstrak_id_expanded
0,"[abstrak, satiyah, pengaruh, faktorfaktor, lat...","[abstrak, satiyah, pengaruh, faktor, latih, ke..."
1,"[tuju, teliti, persepsi, brand, association, l...","[tuju, teliti, persepsi, brand, association, l..."
2,[],[]
3,"[aplikasi, nyata, manfaat, teknologi, informas...","[aplikasi, nyata, manfaat, teknologi, informas..."
4,"[abstrak, teliti, metode, kuantitatif, tekan, ...","[abstrak, teliti, metode, kuantitatif, tekan, ..."
5,"[abstrak, aththaariq, pengaruh, kompetensi, do...","[abstrak, aththaariq, pengaruh, kompetensi, do..."
6,"[abstrak, haryono, arifin, pengaruh, perilaku,...","[abstrak, haryono, arifin, pengaruh, perilaku,..."
7,"[abstrak, dharma, abidin, syahkesimpulan, peng...","[abstrak, dharma, abidin, syahkesimpulan, peng..."
8,"[abstrak, tuju, teliti, identifikasi, variabel...","[abstrak, tuju, teliti, identifikasi, variabel..."
9,"[hasil, teliti, hitung, credit, risk, ratio, t...","[hasil, teliti, hitung, credit, risk, ratio, t..."


In [52]:
# Inisialisasi SpellChecker kosong
spell = SpellChecker(language=None)

# Load kamus Indonesia dari file
with open("00-indonesian-wordlist.lst", "r", encoding="latin-1") as f:
    indo_words = [line.strip() for line in f.readlines()]

spell.word_frequency.load_words(indo_words)

# Fungsi untuk koreksi kata
def correct_word(word):
    corr = spell.correction(word)
    return corr if corr is not None else word

# Terapkan spellcheck ke setiap baris dengan progress bar
corrected_texts = []
for tokens in tqdm(pta_manajemen_df["abstrak_id_expanded"], desc="Spellchecking", unit="row"):
    corrected = [correct_word(word) for word in tokens]
    corrected_texts.append(corrected)

pta_manajemen_df["abstrak_id_spellchecked"] = corrected_texts

Spellchecking: 100%|██████████| 1031/1031 [1:23:53<00:00,  4.88s/row]


In [54]:
print("\nPTA - Abstrak ID (Cek Ejaan):")
pta_manajemen_df[["abstrak_id_expanded", "abstrak_id_spellchecked"]].head(10)


PTA - Abstrak ID (Cek Ejaan):


,abstrak_id_expanded,abstrak_id_spellchecked
0,"[abstrak, satiyah, pengaruh, faktor, latih, ke...","[abstrak, aliyah, pengaruh, faktor, latih, kem..."
1,"[tuju, teliti, persepsi, brand, association, l...","[tuju, teliti, persepsi, band, association, la..."
2,[],[]
3,"[aplikasi, nyata, manfaat, teknologi, informas...","[aplikasi, nyata, manfaat, teknologi, informas..."
4,"[abstrak, teliti, metode, kuantitatif, tekan, ...","[abstrak, teliti, metode, kuantitatif, tekan, ..."
5,"[abstrak, aththaariq, pengaruh, kompetensi, do...","[abstrak, aththaariq, pengaruh, kompetensi, do..."
6,"[abstrak, haryono, arifin, pengaruh, perilaku,...","[abstrak, harmoni, arifin, pengaruh, perilaku,..."
7,"[abstrak, dharma, abidin, syahkesimpulan, peng...","[abstrak, dharma, abidin, syahkesimpulan, peng..."
8,"[abstrak, tuju, teliti, identifikasi, variabel...","[abstrak, tuju, teliti, identifikasi, variabel..."
9,"[hasil, teliti, hitung, credit, risk, ratio, t...","[hasil, teliti, hitung, kredit, disk, patio, t..."


In [61]:
# Pilih kolom yang ingin disimpan
cols_to_save = [
    "abstrak_id",
    "abstrak_id_clean",
    "abstrak_id_tokens",
    "abstrak_id_filtered",
    "abstrak_id_stemmed",
    "abstrak_id_expanded",
    "abstrak_id_spellchecked"
]

# Simpan ke CSV
pta_manajemen_df[cols_to_save].to_csv("pta_processed.csv", index=False, encoding="utf-8-sig")

print("File berhasil disimpan sebagai pta_processed.csv")


File berhasil disimpan sebagai pta_processed.csv


In [57]:
pta_manajemen_df_prs= pd.read_csv("pta_processed.csv")
pta_manajemen_df_prs

,abstrak_id,abstrak_id_clean,abstrak_id_tokens,abstrak_id_filtered,abstrak_id_stemmed,abstrak_id_expanded,abstrak_id_spellchecked
0,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",abstrak satiyah pengaruh faktorfaktor pelatiha...,"['abstrak', 'satiyah', 'pengaruh', 'faktorfakt...","['abstrak', 'satiyah', 'pengaruh', 'faktorfakt...","['abstrak', 'satiyah', 'pengaruh', 'faktorfakt...","['abstrak', 'satiyah', 'pengaruh', 'faktor', '...","['abstrak', 'aliyah', 'pengaruh', 'faktor', 'l..."
1,Tujuan penelitian ini adalah untuk mengetahui ...,tujuan penelitian ini adalah untuk mengetahui ...,"['tujuan', 'penelitian', 'ini', 'adalah', 'unt...","['tujuan', 'penelitian', 'persepsi', 'brand', ...","['tuju', 'teliti', 'persepsi', 'brand', 'assoc...","['tuju', 'teliti', 'persepsi', 'brand', 'assoc...","['tuju', 'teliti', 'persepsi', 'band', 'associ..."
2,NaN,NaN,[],[],[],[],[]
3,Aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata pemanfaatan teknologi informasi...,"['aplikasi', 'nyata', 'pemanfaatan', 'teknolog...","['aplikasi', 'nyata', 'pemanfaatan', 'teknolog...","['aplikasi', 'nyata', 'manfaat', 'teknologi', ...","['aplikasi', 'nyata', 'manfaat', 'teknologi', ...","['aplikasi', 'nyata', 'manfaat', 'teknologi', ..."
4,Abstrak\r\nPenelitian ini menggunakan metode k...,abstrak penelitian ini menggunakan metode kuan...,"['abstrak', 'penelitian', 'ini', 'menggunakan'...","['abstrak', 'penelitian', 'metode', 'kuantitat...","['abstrak', 'teliti', 'metode', 'kuantitatif',...","['abstrak', 'teliti', 'metode', 'kuantitatif',...","['abstrak', 'teliti', 'metode', 'kuantitatif',..."
...,...,...,...,...,...,...,...
1026,ABSTRAK\nPenelitian ini bertujuan untuk menget...,abstrak penelitian ini bertujuan untuk mengeta...,"['abstrak', 'penelitian', 'ini', 'bertujuan', ...","['abstrak', 'penelitian', 'bertujuan', 'perhit...","['abstrak', 'teliti', 'tuju', 'hitung', 'tingk...","['abstrak', 'teliti', 'tuju', 'hitung', 'tingk...","['abstrak', 'teliti', 'tuju', 'hitung', 'tingk..."
1027,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...",abstrak uswatun hasanah pengaruh pelatihan dan...,"['abstrak', 'uswatun', 'hasanah', 'pengaruh', ...","['abstrak', 'uswatun', 'hasanah', 'pengaruh', ...","['abstrak', 'uswatun', 'hasanah', 'pengaruh', ...","['abstrak', 'uswatun', 'hasanah', 'pengaruh', ...","['abstrak', 'uswatun', 'hadanah', 'pengaruh', ..."
1028,ABSTRAK\nTujuan dari penelitian ini adalah unt...,abstrak tujuan dari penelitian ini adalah untu...,"['abstrak', 'tujuan', 'dari', 'penelitian', 'i...","['abstrak', 'tujuan', 'penelitian', 'peran', '...","['abstrak', 'tuju', 'teliti', 'peran', 'servic...","['abstrak', 'tuju', 'teliti', 'peran', 'servic...","['abstrak', 'tuju', 'teliti', 'peran', 'servis..."
1029,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,abstrak penelitian ini bertujuan untuk mengeta...,"['abstrak', 'penelitian', 'ini', 'bertujuan', ...","['abstrak', 'penelitian', 'bertujuan', 'bauran...","['abstrak', 'teliti', 'tuju', 'baur', 'promosi...","['abstrak', 'teliti', 'tuju', 'baur', 'promosi...","['abstrak', 'teliti', 'tuju', 'baur', 'promosi..."


## Index Frequensi

In [59]:
from collections import Counter
import numpy as np

# Gabungkan semua token jadi satu list besar
all_tokens = [word for tokens in pta_manajemen_df["abstrak_id_spellchecked"] for word in tokens]

# Hitung frekuensi kata
word_freq = Counter(all_tokens)

# Ambil 20 kata paling sering muncul
common_words = word_freq.most_common(20)

print("=== 20 Kata Terbanyak ===")
for word, freq in common_words:
    print(f"{word} : {freq}")

# Informasi tambahan
total_words = len(all_tokens)                       # jumlah total kata
unique_words = len(word_freq)                       # jumlah kata unik
avg_word_len = np.mean([len(word) for word in all_tokens])  # rata-rata panjang kata
avg_words_per_data = np.mean([len(tokens) for tokens in pta_manajemen_df["abstrak_id_spellchecked"]])  # rata-rata jumlah kata per data
longest_word = max(all_tokens, key=len)             # kata terpanjang
longest_word_len = len(longest_word)

print("\n=== Statistik Tambahan ===")
print(f"Jumlah total kata       : {total_words}")
print(f"Jumlah kata unik        : {unique_words}")
print(f"Rata-rata panjang kata  : {avg_word_len:.2f} karakter")
print(f"Rata-rata kata per data : {avg_words_per_data:.2f} kata")
print(f"Kata terpanjang         : '{longest_word}' ({longest_word_len} karakter)")


=== 20 Kata Terbanyak ===
pengaruh : 5551
kerja : 5393
teliti : 4381
variabel : 3668
usaha : 2535
signifikan : 2489
uji : 2364
karyawan : 2265
nilai : 1907
hasil : 1784
analisis : 1499
positif : 1321
sampel : 1174
data : 1065
putus : 1040
tuju : 1022
parsial : 1001
metode : 966
simultan : 958
tingkat : 929

=== Statistik Tambahan ===
Jumlah total kata       : 149343
Jumlah kata unik        : 5583
Rata-rata panjang kata  : 6.23 karakter
Rata-rata kata per data : 144.85 kata
Kata terpanjang         : 'mempengaruhikoefisienresponlabasepertikebijakanperusahaanbaiksecarainternalmaupuneksternaldankondisipasar' (105 karakter)


### Simpan semua kata + frekuensi ke CSV


In [60]:
word_freq_df = pd.DataFrame(word_freq.items(), columns=["kata", "jumlah"])
word_freq_df = word_freq_df.sort_values(by="jumlah", ascending=False).reset_index(drop=True)
word_freq_df.to_csv("pta_word_frequency.csv", index=False, encoding="utf-8-sig")

print("\nFile berhasil disimpan sebagai pta_word_frequency.csv.")


File berhasil disimpan sebagai pta_word_frequency.csv.
